# 04. Poisson Model PRO

Modelado probabilístico de goles:
 - Predicción de goles esperados (λ_home, λ_away)
 - Generación de matriz de probabilidades
 - Derivación de probabilidades 1X2

In [19]:
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import poisson

from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss, mean_poisson_deviance
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

dataPath = Path("../data/processed/model_dataset.csv")
fullPath = Path("../data/processed/full_matches.csv")

df = pd.read_csv(dataPath, parse_dates=["date"])
scoresDf = pd.read_csv(
    fullPath,
    parse_dates=["date"],
    usecols=["date", "homeTeam", "awayTeam", "homeScore", "awayScore"],
)

df = df.merge(
    scoresDf,
    on=["date", "homeTeam", "awayTeam"],
    how="left",
    validate="1:1"
)

print("Dataset base de features:", df.shape)
print("Filas con scores disponibles:", df[["homeScore", "awayScore"]].notna().all(axis=1).sum())
df.head()

Dataset base de features: (47769, 26)
Filas con scores disponibles: 47769


,date,homeTeam,awayTeam,eloHome,eloAway,eloDiff,absEloDiff,eloExpectedHomeWin,homeMatchesPlayed,awayMatchesPlayed,homeWinRate,awayWinRate,winRateDiff,homeGoalsForAvg,awayGoalsForAvg,goalsForDiff,homeGoalsAgainstAvg,awayGoalsAgainstAvg,goalsAgainstDiff,attackDiff,defenseDiff,tournamentWeight,neutral,target,homeScore,awayScore
0,1877-03-03,England,Scotland,1496.949971,1506.023694,-9.073723,9.073723,0.486945,5,6,0.200000,0.500000,-0.300000,1.400000,2.166667,-0.766667,1.80,1.166667,0.633333,0.233333,-0.366667,0.3,0,awayWin,1,3
1,1878-03-02,Scotland,England,1511.842486,1494.028302,17.814184,17.814184,0.525614,8,6,0.625000,0.166667,0.458333,2.250000,1.333333,0.916667,1.00,2.000000,-1.000000,0.250000,-0.333333,0.3,0,homeWin,7,2
2,1879-04-05,England,Scotland,1494.183063,1517.511482,-23.328419,23.328419,0.466478,8,10,0.250000,0.700000,-0.450000,1.500000,3.400000,-1.900000,2.50,1.000000,1.500000,0.500000,-0.900000,0.3,0,homeWin,5,4
3,1880-03-13,Scotland,England,1517.086224,1497.384194,19.702030,19.702030,0.528323,12,9,0.666667,0.333333,0.333333,3.416667,1.888889,1.527778,1.25,2.666667,-1.416667,0.750000,-0.638889,0.3,0,homeWin,5,4
4,1880-03-15,Wales,England,1485.529582,1494.554133,-9.024551,9.024551,0.487016,5,10,0.000000,0.300000,-0.300000,0.200000,2.100000,-1.900000,4.00,2.900000,1.100000,-2.700000,1.900000,0.3,0,awayWin,2,3


## 2. Selección de features

In [7]:
# ## 1. Plantilla local de grupos

# %%

#groupsTemplate = {
#    "A": ["Mexico", "South Korea", "South Africa", "UEFA Playoff D winner"],
#    "B": ["Canada", "Switzerland", "Qatar", "UEFA Playoff A winner"],
#    "C": ["Brazil", "Morocco", "Scotland", "Haiti"],
#    "D": ["USA", "Paraguay", "Australia", "UEFA Playoff C winner"],
#    "E": ["Germany", "Ecuador", "Ivory Coast", "Curaçao"],
#    "F": ["Netherlands", "Japan", "Tunisia", "UEFA Playoff B winner"],
#    "G": ["Belgium", "Iran", "Egypt", "New Zealand"],
#    "H": ["Spain", "Uruguay", "Saudi Arabia", "Cape Verde"],
#    "I": ["France", "Senegal", "Norway", "Interconfed Playoff 2 winner"],
#    "J": ["Argentina", "Austria", "Algeria", "Jordan"],
#    "K": ["Portugal", "Colombia", "Uzbekistan", "Interconfed Playoff 1 winner"],
#    "L": ["England", "Croatia", "Panama", "Ghana"],
#}

#print("Grupos plantilla cargados:", {k: len(v) for k, v in groupsTemplate.items()})

In [20]:
# %%
features = [
    "eloDiff",
    "eloExpectedHomeWin",
    "winRateDiff",
    "goalsForDiff",
    "goalsAgainstDiff",
    "attackDiff",
    "defenseDiff",
    "tournamentWeight",
    "neutral"
]

targetHome = "homeScore"
targetAway = "awayScore"

missingFeatures = [col for col in features if col not in df.columns]
missingTargets = [col for col in [targetHome, targetAway] if col not in df.columns]
assert not missingFeatures, f"Faltan features en model_dataset.csv: {missingFeatures}"
assert not missingTargets, f"Faltan targets en df: {missingTargets}"

dfModel = df.dropna(subset=features + [targetHome, targetAway]).copy()

X = dfModel[features]
yHome = dfModel[targetHome].astype(int)
yAway = dfModel[targetAway].astype(int)

print("Shape model:", X.shape)

Shape model: (47769, 9)


## 3. Split temporal (CRÍTICO)


In [21]:
splitDate = "2018-01-01"

trainMask = dfModel["date"] < splitDate
testMask = dfModel["date"] >= splitDate

X_train, X_test = X[trainMask], X[testMask]
yHome_train, yHome_test = yHome[trainMask], yHome[testMask]
yAway_train, yAway_test = yAway[trainMask], yAway[testMask]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (40016, 9)
Test: (7753, 9)


## 4. Modelos Poisson (core)

In [22]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [23]:
homeModel = PoissonRegressor(alpha=0.0001, max_iter=500)
awayModel = PoissonRegressor(alpha=0.0001, max_iter=500)

homeModel.fit(X_train_scaled, yHome_train)
awayModel.fit(X_train_scaled, yAway_train)

PoissonRegressor(alpha=0.0001, max_iter=500)

## 5. Predicción de goles esperados

In [24]:
# %%
lambdaHome = homeModel.predict(X_test_scaled)
lambdaAway = awayModel.predict(X_test_scaled)

print("STD Home:", np.std(lambdaHome))
print("STD Away:", np.std(lambdaAway))

STD Home: 0.762460419640001
STD Away: 0.5868713582846007


## 6. Evaluación (Poisson deviance)

In [25]:
# %%
homeDev = mean_poisson_deviance(yHome_test, lambdaHome)
awayDev = mean_poisson_deviance(yAway_test, lambdaAway)

print("Home deviance:", homeDev)
print("Away deviance:", awayDev)

Home deviance: 1.2798362135698094
Away deviance: 1.1963808650928003


## 7. Función de matriz de probabilidades

In [28]:
# %%
def computeMatchProbs(lambdaHome, lambdaAway, maxGoals=8):
    goals = np.arange(0, maxGoals + 1)

    homePMF = poisson.pmf(goals, lambdaHome)
    awayPMF = poisson.pmf(goals, lambdaAway)

    probMatrix = np.outer(homePMF, awayPMF)

    homeWin = np.tril(probMatrix, -1).sum()
    draw = np.trace(probMatrix)
    awayWin = np.triu(probMatrix, 1).sum()

    return homeWin, draw, awayWin

In [29]:
def dixonColesAdjustment(probMatrix, rho=0.1):
    adjusted = probMatrix.copy()

    adjusted[0,0] *= (1 - rho)
    adjusted[1,0] *= (1 + rho)
    adjusted[0,1] *= (1 + rho)
    adjusted[1,1] *= (1 - rho)

    return adjusted / adjusted.sum()

## 8. Generación de probabilidades 1X2

In [ ]:
maxGoals = 8
goals = np.arange(maxGoals + 1)

homePMF = poisson.pmf(goals[:, None], lambdaHome)
awayPMF = poisson.pmf(goals[:, None], lambdaAway)

# shape: (goals, matches)
homePMF = homePMF.T  # (matches, goals)
awayPMF = awayPMF.T  # (matches, goals)

# Construcción vectorizada de matrices
probMatrices = homePMF[:, :, None] * awayPMF[:, None, :]

# Dixon-Coles vectorizado
rho = 0.05  # provisional (luego lo optimizamos)

probMatrices[:, 0, 0] *= (1 - rho)
probMatrices[:, 1, 0] *= (1 + rho)
probMatrices[:, 0, 1] *= (1 + rho)
probMatrices[:, 1, 1] *= (1 - rho)

# Renormalización
probMatrices = probMatrices / probMatrices.sum(axis=(1,2), keepdims=True)

# Probabilidades 1X2
homeWin = np.tril(probMatrices, -1).sum(axis=(1,2))
draw = np.trace(probMatrices, axis1=1, axis2=2)
awayWin = np.triu(probMatrices, 1).sum(axis=(1,2))

probs = np.vstack([homeWin, draw, awayWin]).T

## Estimación de rho (MUY IMPORTANTE)

In [ ]:
def optimizeRho(probMatrices, yHome, yAway, lambdasHome, lambdasAway):
    rhos = np.linspace(-0.2, 0.2, 21)
    bestRho = 0
    bestLoss = np.inf

    for rho in rhos:
        adjMatrices = probMatrices.copy()

        adjMatrices[:, 0, 0] *= (1 - rho)
        adjMatrices[:, 1, 0] *= (1 + rho)
        adjMatrices[:, 0, 1] *= (1 + rho)
        adjMatrices[:, 1, 1] *= (1 - rho)

        adjMatrices = adjMatrices / adjMatrices.sum(axis=(1,2), keepdims=True)

        # likelihood aproximado
        likelihood = 0
        for i in range(len(yHome)):
            if yHome.iloc[i] <= 8 and yAway.iloc[i] <= 8:
                likelihood += -np.log(adjMatrices[i, yHome.iloc[i], yAway.iloc[i]] + 1e-12)

        if likelihood < bestLoss:
            bestLoss = likelihood
            bestRho = rho

    return bestRho

## 9. Predicción final

In [32]:
from sklearn.metrics import log_loss

logloss = log_loss(trueLabels, probs)
print("LogLoss:", logloss)

print("Balanced Accuracy:", balanced_accuracy_score(trueLabels, predLabels))
print("F1 Macro:", f1_score(trueLabels, predLabels, average="macro"))

LogLoss: 1.7362203383092922
Balanced Accuracy: 0.49340184479045573
F1 Macro: 0.4349917536649251


In [31]:
labels = ["homeWin", "draw", "awayWin"]

predIdx = np.argmax(probs, axis=1)
predLabels = [labels[i] for i in predIdx]

trueLabels = dfModel.loc[testMask, "target"]

accuracy = (np.array(predLabels) == trueLabels.values).mean()

print("Final Accuracy:", accuracy)

Final Accuracy: 0.5958983619244164


In [18]:
# %%
trueLabels = dfModel.loc[testMask, "target"]

accuracy = (predDf["predictedLabel"] == trueLabels).mean()

print("Poisson Accuracy:", accuracy)

Poisson Accuracy: 0.4760737778924287
